In [ ]:
#| default_exp core

# GhApi details

> Detailed information on the GhApi API

In [ ]:
#| export
from fastcore.all import *
from fastspec.spec import SpecParser
from fastspec.oapi import OpenAPIClient, OpFunc, SyncOpFunc, _build_groups, UNSET
from fastspec.transport import AsyncTransport, SyncTransport
from ghapi.gh_spec import spec
from fastspec.errors import APIError

import mimetypes,base64
from collections import Counter
from urllib.parse import quote
from datetime import datetime
import os, shutil, tempfile, subprocess, fnmatch, html

In [ ]:
#| export
_all_ = ['APIError']

In [ ]:
#| hide
from nbdev import *
from IPython.display import Markdown
from datetime import timedelta, timezone
from time import sleep

In [ ]:
#| export
GH_HOST = os.getenv('GH_HOST', "https://api.github.com")
_DOC_URL = 'https://docs.github.com/'

You can set an environment variable named `GH_HOST` to override the default of `https://api.github.com` incase you are running [GitHub Enterprise](https://github.com/enterprise)(GHE).  However, this library has not been tested on GHE, so proceed at your own risk.

In [ ]:
#| export
pspec = SpecParser.from_dict(spec)

In [ ]:
#| export
_binary_cts = ('octet-stream', 'zip', 'gzip', 'tar', 'image/', 'audio/', 'video/')


class GhTransport(AsyncTransport):
    "Async transport converting JSON responses to `AttrDict`s and tracking rate-limit and response headers."
    def __init__(self, debug=None, limit_cb=None, **kwargs):
        super().__init__(**kwargs)
        self.debug,self.limit_cb,self.limit_rem = debug,limit_cb,5000
        self.recv_hdrs = {}

    @staticmethod
    def _decode(resp):
        "Like `AsyncTransport._decode`, but GitHub media types (e.g. diffs, shas) decode to `str` unless truly binary"
        res = AsyncTransport._decode(resp)
        ct = resp.headers.get('content-type') or ''
        if isinstance(res, bytes) and not any(t in ct for t in _binary_cts): res = res.decode()
        return res

    def _pre(self, method, url, kwargs):
        debug = self.debug or (print_summary if os.getenv('GHAPI_DEBUG') else None)
        if debug: debug(method, url, kwargs)

    def _post(self, resp, raw):
        self.recv_hdrs = resp.headers
        if 'X-RateLimit-Remaining' in resp.headers:
            newlim = resp.headers['X-RateLimit-Remaining']
            if self.limit_cb is not None and newlim != self.limit_rem: self.limit_cb(int(newlim), int(resp.headers['X-RateLimit-Limit']))
            self.limit_rem = newlim
        if raw: return resp
        res = self._decode(resp)
        return dict2obj(res) if isinstance(res, (dict,list)) else res

    async def request(self, method, url, *, raw=False, **kwargs):
        self._pre(method, url, kwargs)
        return self._post(await super().request(method, url, raw=True, **kwargs), raw)


class GhSyncTransport(SyncTransport, GhTransport):
    "Sync twin of `GhTransport`: same debug, header, and rate-limit handling over a blocking `SyncTransport`"
    def request(self, method, url, *, raw=False, **kwargs):
        self._pre(method, url, kwargs)
        return self._post(SyncTransport.request(self, method, url, raw=True, **kwargs), raw)

In [ ]:
#| export
_docroot = 'https://docs.github.com/rest/reference/'

In [ ]:
#| export
def print_summary(method, url, kwargs):
    "Debug callback for `GhApi(debug=...)`: print each request with the token (if any) removed"
    hdrs = {k:v for k,v in (kwargs.get('headers') or {}).items() if k.lower()!='authorization'}
    print(method, url, {**kwargs, 'headers':hdrs})

## GhApi -

In [ ]:
#| export
class GhApi(OpenAPIClient):
    "GitHub API client. Endpoint groups (`issues`, `pulls`, ...) are generated per-instance from GitHub's OpenAPI metadata, so the class shows only convenience methods -- inspect a live instance, e.g. `doc(GhApi())`, to see the full API."
    def __init__(self, owner=None, repo=None, token=None, jwt_token=None, debug=None, limit_cb=None, gh_host=None,  # chkstyle: ignore-node
                 authenticate=True, timeout=60.0, sync=False, **kwargs):
        self.headers = { 'Accept': 'application/vnd.github.v3+json' }
        if authenticate:
            token = token or os.getenv('GITHUB_TOKEN', None)
            jwt_token = jwt_token or os.getenv('GITHUB_JWT_TOKEN', None)
            if jwt_token: self.headers['Authorization'] = 'Bearer ' + jwt_token
            elif token: self.headers['Authorization'] = 'token ' + token
            else: warn('Neither GITHUB_TOKEN nor GITHUB_JWT_TOKEN found: running as unauthenticated')
        if owner: kwargs['owner'] = owner
        if repo:  kwargs['repo' ] = repo
        self.token,self.gh_host = token,gh_host or GH_HOST
        tcls,fcls = (GhSyncTransport,SyncOpFunc) if sync else (GhTransport,OpFunc)
        self.transport = tcls(debug=debug, limit_cb=limit_cb, timeout=timeout, base_headers=self.headers)
        self.ops = [fcls(o, self.transport, self.gh_host, defaults=kwargs) for o in pspec.ops]
        self.func_dict = {f'{o.path}:{o.verb.upper()}':o for o in self.ops}
        self.groups = _build_groups(self.ops)
        for k,v in self.groups.items(): setattr(self, k, v)

    def __call__(self, path:str, verb:str=None, headers:dict=None, route:dict=None, query:dict=None, data=None):
        "Call a fully specified `path` (or full URL) using HTTP `verb` directly (returns an awaitable on an async client)"
        if verb is None: verb = 'POST' if data else 'GET'
        if route: path = path.format(**{k:quote(str(v), safe='') for k,v in route.items()})
        if not path.startswith(('http://', 'https://')): path = self.gh_host + path
        kw = dict(content=data) if isinstance(data, (bytes,str)) else dict(json_data=data)
        return self.transport.request(verb, path, headers=headers, params=query, **kw)

    @property
    def debug(self): return self.transport.debug
    @debug.setter
    def debug(self, v): self.transport.debug = v
    @property
    def limit_rem(self): return self.transport.limit_rem
    @property
    def recv_hdrs(self): return self.transport.recv_hdrs

    def _repr_markdown_(self): return "\n".join(f"- [{o}]({_docroot + o.replace('_', '-')})" for o in sorted(self.groups))

    def __getitem__(self, k):
        "Lookup an endpoint by path and verb (which defaults to 'GET')"
        a,b = k if isinstance(k,tuple) else (k,'GET')
        return self.func_dict[f'{a}:{b.upper()}']

In [ ]:
#| hide
token = os.environ['GITHUB_TOKEN']

### Access by path

In [ ]:
show_doc(GhApi.__call__)

<div class="prose" markdown="1">

---

[source](https://github.com/fastai/ghapi/blob/main/ghapi/core.py#L110){target="_blank" style="float:right; font-size:smaller"}

### GhApi.__call__

```python
def __call__(
    path:str, verb:str=None, headers:dict=None, route:dict=None, query:dict=None, data:NoneType=None
):
```

*Call a fully specified `path` (or full URL) using HTTP `verb` directly (returns an awaitable on an async client)*

</div>

In [ ]:
api = GhApi()

You can call a `GhApi` object as a function, passing in the path to the endpoint, the HTTP verb, and any route, query parameter, or post data parameters as required.

In [ ]:
await api('/repos/{owner}/{repo}/git/ref/{ref}', 'GET', route=dict(
    owner='fastai', repo='ghapi-test', ref='heads/master'))

<div class="prose" markdown="1">

```python
{ 'node_id': 'MDM6UmVmMzE1NzEyNTg4OnJlZnMvaGVhZHMvbWFzdGVy',
  'object': { 'sha': 'b72d6c87a9237ca3c26298a64a6acf06217ace4a',
              'type': 'commit',
              'url': 'https://api.github.com/repos/fastai/ghapi-test/git/commits/b72d6c87a9237ca3c26298a64a6acf06217ace4a'},
  'ref': 'refs/heads/master',
  'url': 'https://api.github.com/repos/fastai/ghapi-test/git/refs/heads/master'}
```

</div>

In [ ]:
show_doc(GhApi.__getitem__)

<div class="prose" markdown="1">

---

[source](https://github.com/fastai/ghapi/blob/main/ghapi/core.py#L129){target="_blank" style="float:right; font-size:smaller"}

### GhApi.__getitem__

```python
def __getitem__(
    k
):
```

*Lookup an endpoint by path and verb (which defaults to 'GET')*

</div>

You can access endpoints by indexing into the object. When using the API this way, you do not need to specify what type of parameter (route, query, or post data) is being used. This is, therefore, the same call as above:

In [ ]:
await api['/repos/{owner}/{repo}/git/ref/{ref}'](owner='fastai', repo='ghapi-test', ref='heads/master')

<div class="prose" markdown="1">

```python
{ 'node_id': 'MDM6UmVmMzE1NzEyNTg4OnJlZnMvaGVhZHMvbWFzdGVy',
  'object': { 'sha': 'b72d6c87a9237ca3c26298a64a6acf06217ace4a',
              'type': 'commit',
              'url': 'https://api.github.com/repos/fastai/ghapi-test/git/commits/b72d6c87a9237ca3c26298a64a6acf06217ace4a'},
  'ref': 'refs/heads/master',
  'url': 'https://api.github.com/repos/fastai/ghapi-test/git/refs/heads/master'}
```

</div>

### Media types

For some endpoints GitHub lets you specify a [media type](https://docs.github.com/en/rest/overview/media-types) the for response data, using the `Accept` header. If you choose a media type that is not JSON formatted (for instance `application/vnd.github.v3.sha`) then the call to the `GhApi` object will return a string instead of an object.

In [ ]:
await api('/repos/{owner}/{repo}/commits/{ref}', 'GET', route=dict(owner='fastai', repo='ghapi-test', ref='refs/heads/master'),
    headers={'Accept': 'application/vnd.github.VERSION.sha'})

'b72d6c87a9237ca3c26298a64a6acf06217ace4a'

### Rate limits

GitHub has various [rate limits](https://docs.github.com/rest/overview/resources-in-the-rest-api#rate-limiting) for their API. After each call, the response includes information about how many requests are remaining in the hourly quota. If you'd like to add alerts, or indications showing current quota usage, you can register a callback with `GhApi` by passing a callable to the `limit_cb` parameter. This callback will be called whenever the amount of quota used changes. It will be called with two arguments: the new quota remaining, and the total hourly quota.

In [ ]:
def _f(rem,quota): print(f"Quota remaining: {rem} of {quota}")

api = GhApi(limit_cb=_f)
(await api['/repos/{owner}/{repo}/git/ref/{ref}'](owner='fastai', repo='ghapi-test', ref='heads/master')).ref

Quota remaining: 4931 of 5000


'refs/heads/master'

You can always get the remaining quota from the `limit_rem` attribute:

In [ ]:
api.limit_rem

'4931'

### Sync usage

Everything in `ghapi` is async by default. For code that can't (or shouldn't) be async, pass `sync=True` to get a client whose endpoint calls block and return results directly, with the same groups, names, and signatures:

In [ ]:
sapi = GhApi(owner='fastai', repo='ghapi-test', sync=True)
test_eq(sapi.repos.get().name, 'ghapi-test')
test_eq(sapi['/repos/{owner}/{repo}/git/ref/{ref}'](owner='fastai', repo='ghapi-test', ref='heads/master').object.type, 'commit')

Only the generated endpoints are sync on such a client: the convenience methods below (`read_issue`, `create_gist`, ...) are written as async code, so they stay awaitable-only. To use those from sync code, call them on a regular async client via `fastcore.net.run_sync` (a stdlib-only bridge that works even inside Jupyter), e.g. `run_sync(GhApi().read_issue(205))`. Pagination has sync twins, `sync_paged` and `sync_pages` (see `page`). And for a quick one-off endpoint call, `call_gh` wraps constructing a sync client and calling one operation:

In [ ]:
#| export
def call_gh(op:str, *args, token=None, **kwargs):
    "Call one GitHub operation `op` (e.g. `'repos.get'`) on a fresh sync client; handy for one-off calls from sync code"
    f = GhApi(token=token, sync=True)
    for part in op.split('.'): f = getattr(f, part)
    return f(*args, **kwargs)

In [ ]:
test_eq(call_gh('repos.get', owner='fastai', repo='ghapi-test').name, 'ghapi-test')

## Operations

Instead of passing a path to `GhApi`, you will more often use the operation methods provided in the API's operation groups, which include documentation, signatures, and auto-complete.

If you provide `owner` and/or `repo` to the constructor, they will be automatically inserted into any calls which use them (except when calling `GhApi` as a function). You can also pass any other arbitrary keyword arguments you like to have them used as defaults for any relevant calls.

You must include a GitHub API token if you need to access any authenticated endpoints. If don't pass the `token` param, then your `GITHUB_TOKEN` environment variable will be used, if available.

In [ ]:
api = GhApi(owner='AnswerDotAI', repo='ghapi-test', token=token)

### Operation groups

The following groups of endpoints are provided, which you can list at any time along with a link to documentation for all endpoints in that group, by displaying the `GhApi` object:

In [ ]:
api

<div class="prose" markdown="1">

- [actions](https://docs.github.com/rest/reference/actions)
- [activity](https://docs.github.com/rest/reference/activity)
- [agent_tasks](https://docs.github.com/rest/reference/agent-tasks)
- [agents](https://docs.github.com/rest/reference/agents)
- [api_insights](https://docs.github.com/rest/reference/api-insights)
- [apps](https://docs.github.com/rest/reference/apps)
- [billing](https://docs.github.com/rest/reference/billing)
- [campaigns](https://docs.github.com/rest/reference/campaigns)
- [checks](https://docs.github.com/rest/reference/checks)
- [classroom](https://docs.github.com/rest/reference/classroom)
- [code_quality](https://docs.github.com/rest/reference/code-quality)
- [code_scanning](https://docs.github.com/rest/reference/code-scanning)
- [code_security](https://docs.github.com/rest/reference/code-security)
- [codes_of_conduct](https://docs.github.com/rest/reference/codes-of-conduct)
- [codespaces](https://docs.github.com/rest/reference/codespaces)
- [copilot](https://docs.github.com/rest/reference/copilot)
- [copilot_spaces](https://docs.github.com/rest/reference/copilot-spaces)
- [credentials](https://docs.github.com/rest/reference/credentials)
- [dependabot](https://docs.github.com/rest/reference/dependabot)
- [dependency_graph](https://docs.github.com/rest/reference/dependency-graph)
- [emojis](https://docs.github.com/rest/reference/emojis)
- [enterprise_team_memberships](https://docs.github.com/rest/reference/enterprise-team-memberships)
- [enterprise_team_organizations](https://docs.github.com/rest/reference/enterprise-team-organizations)
- [enterprise_teams](https://docs.github.com/rest/reference/enterprise-teams)
- [gists](https://docs.github.com/rest/reference/gists)
- [git](https://docs.github.com/rest/reference/git)
- [gitignore](https://docs.github.com/rest/reference/gitignore)
- [hosted_compute](https://docs.github.com/rest/reference/hosted-compute)
- [interactions](https://docs.github.com/rest/reference/interactions)
- [issues](https://docs.github.com/rest/reference/issues)
- [licenses](https://docs.github.com/rest/reference/licenses)
- [markdown](https://docs.github.com/rest/reference/markdown)
- [meta](https://docs.github.com/rest/reference/meta)
- [migrations](https://docs.github.com/rest/reference/migrations)
- [oidc](https://docs.github.com/rest/reference/oidc)
- [orgs](https://docs.github.com/rest/reference/orgs)
- [packages](https://docs.github.com/rest/reference/packages)
- [private_registries](https://docs.github.com/rest/reference/private-registries)
- [projects](https://docs.github.com/rest/reference/projects)
- [pulls](https://docs.github.com/rest/reference/pulls)
- [rate_limit](https://docs.github.com/rest/reference/rate-limit)
- [reactions](https://docs.github.com/rest/reference/reactions)
- [repos](https://docs.github.com/rest/reference/repos)
- [search](https://docs.github.com/rest/reference/search)
- [secret_scanning](https://docs.github.com/rest/reference/secret-scanning)
- [security_advisories](https://docs.github.com/rest/reference/security-advisories)
- [teams](https://docs.github.com/rest/reference/teams)
- [users](https://docs.github.com/rest/reference/users)

</div>

In [ ]:
api.codes_of_conduct

<div class="prose" markdown="1">

- [codes_of_conduct.get_all_codes_of_conduct](https://docs.github.com/rest/codes-of-conduct/codes-of-conduct#get-all-codes-of-conduct)(): *Get all codes of conduct*
- [codes_of_conduct.get_conduct_code](https://docs.github.com/rest/codes-of-conduct/codes-of-conduct#get-a-code-of-conduct)(key): *Get a code of conduct*

</div>

### Calling endpoints

The GitHub API's endpoint names generally start with a verb like "get", "list", "delete", "create", etc, followed `_`, then by a noun such as "ref", "webhook", "issue", etc.

Each endpoint has a different signature, which you can see by using <kbd>Shift</kbd>-<kbd>Tab</kbd> in Jupyter, or by just printing the endpoint object (which also shows a link to the GitHub docs):

In [ ]:
print(api.repos.create_webhook)

repos.create_webhook(name: str = UNSET, config: dict = UNSET, owner: str = 'AnswerDotAI', repo: str = 'ghapi-test', events: list = ['push'], active: bool = True)
https://docs.github.com/rest/repos/webhooks#create-a-repository-webhook


Displaying an endpoint object in Jupyter also provides a formatted summary and link to the official GitHub documentation:

In [ ]:
api.repos.create_webhook

<div class="prose" markdown="1">

Create a repository webhook

Docs: https://docs.github.com/rest/repos/webhooks#create-a-repository-webhook

Parameters:
- name (str, optional): Use `web` to create a webhook. Default: `web`. This parameter only accepts the value `web`.
- config (dict, optional): Key/value pairs to provide settings for this webhook.
- owner (str, default: 'AnswerDotAI'): The account owner of the repository. The name is not case sensitive.
- repo (str, default: 'ghapi-test'): The name of the repository without the `.git` extension. The name is not case sensitive.
- events (list, default: ['push']): Determines what [events](https://docs.github.com/webhooks/event-payloads) the hook is triggered for.
- active (bool, default: True): Determines if notifications are sent when the webhook is triggered. Set to `true` to send notifications.

</div>

Endpoint objects are called using standard Python method syntax:

In [ ]:
ref = await api.git.get_ref('heads/master')
test_eq(ref.object.type, 'commit')

Information about the endpoint are available as attributes:

In [ ]:
api.git.get_ref.path,api.git.get_ref.verb

('/repos/{owner}/{repo}/git/ref/{ref}', 'GET')

You can get a list of all endpoints available in a group, along with a link to documentation for each, by viewing the group:

In [ ]:
api.git

<div class="prose" markdown="1">

- [git.create_blob](https://docs.github.com/rest/git/blobs#create-a-blob)(content, owner, repo, encoding): *Create a blob*
- [git.get_blob](https://docs.github.com/rest/git/blobs#get-a-blob)(file_sha, owner, repo): *Get a blob*
- [git.create_commit](https://docs.github.com/rest/git/commits#create-a-commit)(message, tree, parents, author, committer, signature, owner, repo): *Create a commit*
- [git.get_commit](https://docs.github.com/rest/git/commits#get-a-commit-object)(commit_sha, owner, repo): *Get a commit object*
- [git.list_matching_refs](https://docs.github.com/rest/git/refs#list-matching-references)(ref, owner, repo): *List matching references*
- [git.get_ref](https://docs.github.com/rest/git/refs#get-a-reference)(ref, owner, repo): *Get a reference*
- [git.create_ref](https://docs.github.com/rest/git/refs#create-a-reference)(ref, sha, owner, repo): *Create a reference*
- [git.update_ref](https://docs.github.com/rest/git/refs#update-a-reference)(ref, sha, owner, repo, force): *Update a reference*
- [git.delete_ref](https://docs.github.com/rest/git/refs#delete-a-reference)(ref, owner, repo): *Delete a reference*
- [git.create_tag](https://docs.github.com/rest/git/tags#create-a-tag-object)(tag, message, object, type, tagger, owner, repo): *Create a tag object*
- [git.get_tag](https://docs.github.com/rest/git/tags#get-a-tag)(tag_sha, owner, repo): *Get a tag*
- [git.create_tree](https://docs.github.com/rest/git/trees#create-a-tree)(tree, base_tree, owner, repo): *Create a tree*
- [git.get_tree](https://docs.github.com/rest/git/trees#get-a-tree)(tree_sha, recursive, owner, repo): *Get a tree*

</div>

For "list" endpoints, the noun will be a plural form, e.g.:

In [ ]:
#| hide
for hook in await api.repos.list_webhooks(): await api.repos.delete_webhook(hook.id)

In [ ]:
hooks = await api.repos.list_webhooks()
test_eq(len(hooks), 0)

You can pass dicts, lists, etc. directly, where they are required for GitHub API endpoints:

In [ ]:
url = 'https://example.com'
cfg = dict(url=url, content_type='json', secret='XXX')
hook = await api.repos.create_webhook(config=cfg, events=['ping'])
test_eq(hook.config.url, url)

Let's confirm that our new webhook has been created:

In [ ]:
hooks = await api.repos.list_webhooks()
test_eq(len(hooks), 1)
test_eq(hooks[0].events, ['ping'])

Finally, we can delete our new webhook:

In [ ]:
await api.repos.delete_webhook(hooks[0].id)

### Convenience functions

In [ ]:
#| export
def date2gh(dt:datetime)->str:
    "Convert `dt` (which is assumed to be in UTC time zone) to a format suitable for GitHub API operations"
    return f'{dt.replace(microsecond=0).isoformat()}Z'

The GitHub API assumes that dates will be in a specific string format. `date2gh` converts Python standard `datetime` objects to that format. For instance, to find issues opened in the 'fastcore' repo in the last 4 weeks:

In [ ]:
dt = date2gh(datetime.now(timezone.utc) - timedelta(weeks=4))
issues = await GhApi('fastai').issues.list_for_repo(repo='fastcore', since=dt)
len(issues)

2

In [ ]:
#| export
def gh2date(dtstr:str)->datetime:
    "Convert date string `dtstr` received from a GitHub API operation to a UTC `datetime`"
    return datetime.fromisoformat(dtstr.replace('Z', ''))

In [ ]:
# created = issues[0].created_at
# print(created, '->', gh2date(created))

You can set the `debug` attribute to any callable to intercept all requests -- it's called with the method, URL, and request kwargs before each request. `print_summary` is provided for this purpose (it prints each request with the auth token removed), and setting the `GHAPI_DEBUG` environment variable enables it globally:

In [ ]:
api.debug=print_summary
(await api.codes_of_conduct.get_all_codes_of_conduct())[0]
api.debug=None

GET https://api.github.com/codes_of_conduct {'headers': {}, 'params': {}, 'json_data': None}


## Convenience methods

Some methods in the GitHub API are a bit clunky or unintuitive. In these situations we add convenience methods to `GhApi` to make things simpler. There are also some multi-step processes in the GitHub API that `GhApi` provide convenient wrappers for. The methods currently available are shown below; do not hesitate to [create an issue](https://github.com/fastai/ghapi-test/issues) or pull request if there are other processes that you'd like to see supported better.

In [ ]:
#| export
img_md_pat = re.compile(r'!\[(?P<alt>.*?)\]\((?P<url>[^\s]+)\)')

def _run_subp(cmd): 
    r = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(r.stderr)

@patch
async def create_gist(self:GhApi, description, content, filename='gist.txt', public=False, img_paths=None):
    'Create a gist, optionally with images where each md img url will be placed with img upload urls.'
    gist = await self.gists.create(description=description, public=public, files={filename: {"content": content}})
    if not img_paths: return gist
    with tempfile.TemporaryDirectory() as clone_dir:
        token = self.headers['Authorization'].split('token ')[1]
        _run_subp(['git', 'clone', f'https://{token}@gist.github.com/{gist.id}.git', clone_dir])
        clone_dir, img_paths = Path(clone_dir), L(img_paths).map(Path)
        for o in img_paths: shutil.copy2(o, clone_dir/o.name)
        _run_subp(['git', '-C', clone_dir, 'add', '.'])
        _run_subp(['git', '-C', clone_dir, 'commit', '-m', 'Add images'])
        _run_subp(['git', '-C', clone_dir, 'push'])    
    updated_gist = await self.gists.get(gist.id)
    img_urls = {o.name: updated_gist.files[o.name].raw_url for o in img_paths}
    content = img_md_pat.sub(lambda m: f"![{m['alt']}]({img_urls.get(m['url'], m['url'])})", content)
    return await self.gists.update(gist.id, files={filename:{'content':content}})

In [ ]:
gist = await api.create_gist("some description", "some content")
print(gist.html_url)

https://gist.github.com/jph00/d68e0a59cd9fc8e984797d29898b8317


In [ ]:
gist.files['gist.txt'].content

'some content'

In [ ]:
gist = await api.create_gist("some description", "some image\n\n![image](puppy.jpg)", 'gist.md', img_paths=['../puppy.jpg'])
print(gist.html_url)

https://gist.github.com/jph00/929068e9fa15a998e3c58bbb54e1bd24


In [ ]:
gist.files['gist.md'].content

'some image\n\n![image](https://gist.githubusercontent.com/jph00/929068e9fa15a998e3c58bbb54e1bd24/raw/c7f420c839f58c6ac0c05f1116317645d31d7e80/puppy.jpg)'

Note that if you want to create a gist with multiple files, call the GitHub API directly, e.g.:

```python
api.gists.create("some description", files={"f1.txt": {"content": "my content"}, ...})
```

In [ ]:
#| export
@patch
def load_gist(self:GhApi, gist_id:str):
    "Retrieve a gist by id, or by `user/id` (as it appears in gist URLs); coro if async client"
    if '/' in gist_id: *_,user,gist_id = gist_id.split('/')
    else: user = None
    return self.gists.get(gist_id, user=user)

@patch
def gist_file(self:GhApi, gist_id:str):
    "Get the first file from a gist; coro if async client"
    return then(self.load_gist(gist_id), ~Self.files.values(), first)

@patch
async def update_gist(self:GhApi, gist_id:str, content:str):
    "Update the first file in a gist with new content"
    if '/' in gist_id: *_,_,gist_id = gist_id.split('/')
    gist = await self.gists.get(gist_id)
    fname = first(gist.files.keys())
    return (await self.gists.update(gist_id, files={fname: {'content': content}})).html_url

`load_gist` accepts a bare gist id or a `user/id` string (as it appears in a gist's URL). `gist_file` and `update_gist` work with the first file in a gist; for gists with multiple files, use `api.gists.get`/`api.gists.update` directly. `load_gist` and `gist_file` follow the client's mode: on an async client they return awaitables as usual, while on a `GhApi(sync=True)` client they return results directly (`gist_file` does this via fastcore's `then`).


In [ ]:
gistid = 'jph00/e7cfd4ded593e8ef6217e78a0131960c'
loaded = await api.load_gist(gistid)
test_eq(loaded.id, 'e7cfd4ded593e8ef6217e78a0131960c')

gfile = await api.gist_file(gistid)
assert gfile.content

sapi = GhApi(sync=True)
test_eq(sapi.load_gist(gistid).id, 'e7cfd4ded593e8ef6217e78a0131960c')
assert sapi.gist_file(gistid).content

g = await api.create_gist('update_gist test', 'v1')
url = await api.update_gist(g.id, 'v2')
test_eq(url, g.html_url)
for _ in range(20):
    content = first((await api.gists.get(g.id)).files.values()).content
    if content == 'v2': break
    sleep(1)
test_eq(content, 'v2')
await api.gists.delete(g.id)

### Releases

In [ ]:
#| export
@patch
async def delete_release(self:GhApi, release):
    "Delete a release and its associated tag"
    await self.repos.delete_release(release.id)
    await self.git.delete_ref(f'tags/{release.tag_name}')

In [ ]:
#| hide
for rel in await api.repos.list_releases(): await api.delete_release(rel)

In [ ]:
#| export
@patch
async def upload_file(self:GhApi, rel, fn):
    "Upload `fn` to endpoint for release `rel`"
    fn = Path(fn)
    url = rel.upload_url.replace('{?name,label}','')
    mime = mimetypes.guess_type(fn, False)[0] or 'application/octet-stream'
    return await self(url, 'POST', headers={'Content-Type':mime}, query = {'name':fn.name}, data=fn.read_bytes())

In [ ]:
#| export
@patch
async def create_release(self:GhApi, tag_name, branch='master', name=None, body='',
    draft=False, prerelease=False, files=None, make_latest=UNSET):
    "Wrapper for `GhApi.repos.create_release` which also uploads `files`"
    if name is None: name = 'v'+tag_name
    rel = await self.repos.create_release(tag_name, target_commitish=branch, name=name, body=body,
        draft=draft, prerelease=prerelease, make_latest=make_latest)
    for file in listify(files): await self.upload_file(rel, file)
    return rel

Creating a release and attaching files to it is normally a multi-stage process, so `create_release` wraps this up for you. It takes the same arguments as [`repos.create_release`](https://docs.github.com/rest/reference/repos#create-a-release), along with `files`, which can contain a single file name, or a list of file names to upload to your release. Arguments left at their defaults (such as `make_latest`) are omitted from the API call, so GitHub's own defaults apply; pass `make_latest='false'` when releasing an update to an older version, so it doesn't take over the repo's "Latest" marker:

In [ ]:
#| eval: false
rel = await api.create_release('0.0.1', files=['../README.md'])
test_eq(rel.name, 'v0.0.1')

In [ ]:
#| eval: false
for _ in range(20):
    rels = await api.repos.list_releases()
    if len(rels) >= 1: break
    sleep(0.5)
test_eq(len(rels), 1)


We can check that our file has been uploaded; GitHub refers to them as "assets":

In [ ]:
#| eval: false
assets = await api.repos.list_release_assets(rels[0].id)
test_eq(assets[0].name, 'README.md')

In [ ]:
show_doc(GhApi.delete_release)

<div class="prose" markdown="1">

---

[source](https://github.com/fastai/ghapi/blob/main/ghapi/core.py#L199){target="_blank" style="float:right; font-size:smaller"}

### GhApi.delete_release

```python
async def delete_release(
    release
):
```

*Delete a release and its associated tag*

</div>

### Branches and tags

In [ ]:
#| export
@patch
async def list_tags(self:GhApi, prefix:str=''):
    "List all tags, optionally filtered to those starting with `prefix`"
    return await self.git.list_matching_refs(f'tags/{prefix}')

With no `prefix`, all tags are listed.

In [ ]:
#| eval: false
test_eq(len(await api.list_tags()), 1)

Using the full tag name will return just that tag.

In [ ]:
#| eval: false
test_eq(len(await api.list_tags(rel.tag_name)), 1)

In [ ]:
#| export
@patch
async def list_branches(self:GhApi, prefix:str=''):
    "List all branches, optionally filtered to those starting with `prefix`"
    return await self.git.list_matching_refs(f'heads/{prefix}')

Branches can be listed in the exactly the same way as tags.

In [ ]:
test_eq(len(await api.list_branches('master')), 1)

We can delete our release and confirm that it is removed:

In [ ]:
#| eval: false
await api.delete_release(rels[0])
test_eq(len(await api.repos.list_releases()), 0)

In [ ]:
#| export
# See https://stackoverflow.com/questions/9765453
EMPTY_TREE_SHA = '4b825dc642cb6eb9a060e54bf8d69288fbee4904'

In [ ]:
# #| hide
# #not working
# #| export
# @patch
# def create_branch_empty(self:GhApi, branch):
#     c = self.git.create_commit(f'create {branch}', EMPTY_TREE_SHA)
#     return self.git.create_ref(f'refs/heads/{branch}', c.sha)

In [ ]:
#| export
@patch
async def create_branch_empty(self:GhApi, branch):
    t = await self.git.create_tree(base_tree=EMPTY_TREE_SHA, tree = [dict(
        path='.dummy', content='ignore me', mode='100644', type='blob')])
    c = await self.git.create_commit(f'create {branch}', t.sha)
    return await self.git.create_ref(f'refs/heads/{branch}', c.sha)

In [ ]:
ref = await api.create_branch_empty("testme")
test_eq(len(await api.list_branches('testme')), 1)

In [ ]:
#| export
@patch
async def delete_tag(self:GhApi, tag:str):
    "Delete a tag"
    return await self.git.delete_ref(f'tags/{tag}')

In [ ]:
#| export
@patch
async def delete_branch(self:GhApi, branch:str):
    "Delete a branch"
    return await self.git.delete_ref(f'heads/{branch}')

In [ ]:
await api.delete_branch('testme')
for _ in range(20):
    if not await api.list_branches('testme'): break
    sleep(0.5)
test_eq(len(await api.list_branches('testme')), 0)


In [ ]:
#| export
@patch
async def get_branch(self:GhApi, branch=None):
    branch = branch or (await self.repos.get()).default_branch
    return (await self.list_branches(branch))[0]

### Content (git files)

In [ ]:
#| export
@patch
async def list_files(self:GhApi, branch=None):
    ref = await self.get_branch(branch)
    res = (await self.git.get_tree(ref.object.sha)).tree
    return {o.path:o for o in res}

In [ ]:
files = await api.list_files()
files['README.md']

<div class="prose" markdown="1">

```python
{ 'mode': '100644',
  'path': 'README.md',
  'sha': 'eaea0f2698e76c75602058bf4e2e9fd7940ac4e3',
  'size': 72,
  'type': 'blob',
  'url': 'https://api.github.com/repos/AnswerDotAI/ghapi-test/git/blobs/eaea0f2698e76c75602058bf4e2e9fd7940ac4e3'}
```

</div>

In [ ]:
#| export
@patch
async def get_content(self:GhApi, path):
    res = await self.repos.get_content(path)
    return base64.b64decode(res.content)

In [ ]:
readme = (await api.get_content('README.md')).decode()
assert 'ghapi' in readme

In [ ]:
#| export
@patch
async def create_or_update_file(self:GhApi, path, message, committer, author, content=None, sha=None, branch=''):
    if not branch: branch = (await self.repos.get())['default_branch']
    if not isinstance(content,bytes): content = content.encode()
    content = base64.b64encode(content).decode()
    kwargs = {'sha':sha} if sha else {}
    return await self.repos.create_or_update_file_contents(path, message, content=content,
        branch=branch, committer=committer or {}, author=author or {}, **kwargs)

In [ ]:
#| export
@patch
async def create_file(self:GhApi, path, message, committer, author, content=None, branch=None):
    if not branch: branch = (await self.repos.get())['default_branch']
    return await self.create_or_update_file(path, message, branch=branch, committer=committer, content=content, author=author)

In [ ]:
person = dict(name="Monalisa Octocat", email="octocat@github.com")
res = await api.create_file(path='foo', message="Create foo", content="foobar", committer=person, author=person)
test_eq('foobar', (await api.get_content('foo')).decode())

In [ ]:
#| export
@patch
async def delete_file(self:GhApi, path, message, committer, author, sha=None, branch=None):
    if not branch: branch = (await self.repos.get())['default_branch']
    if sha is None: sha = (await self.list_files())[path].sha
    return await self.repos.delete_file(path, message=message, sha=sha, branch=branch, committer=committer, author=author)

In [ ]:
await api.delete_file('foo', 'delete foo', committer=person, author=person)
assert 'foo' not in await api.list_files()

In [ ]:
#| export
@patch
async def update_contents(self:GhApi, path, message, committer, author, content, sha=None, branch=None):
    if not branch: branch = (await self.repos.get())['default_branch']
    if sha is None: sha = (await self.list_files())[path].sha
    return await self.create_or_update_file(path, message, committer=committer, author=author, content=content, sha=sha, branch=branch)

In [ ]:
res = await api.update_contents(path='README.md', message="Update README", committer=person, author=person, content=readme+"foobar")
res.content.size

78

In [ ]:
readme = (await api.get_content('README.md')).decode()
assert 'foobar' in readme
await api.update_contents('README.md', "Revert README", committer=person, author=person, content=readme[:-6]);

In [ ]:
api = GhApi(token=token)

In [ ]:
# Pinned to a fixed commit (not "main") so the file count below stays deterministic as fastcore evolves
owner, repo, branch = "AnswerDotAI", "fastcore", "6c22f5d5a79cb8c9edbd51847d2e7b56c92727a3"

Repo files can be filtered using [fnmatch](https://docs.python.org/3/library/fnmatch.html) Unix shell-style wildcards.

In [ ]:
#| export
def _find_matches(path, pats):
    "Returns matched patterns"
    return L(pats).filter(lambda p: fnmatch.fnmatch(path, p))

In [ ]:
_find_matches('README.md', ['*.py', '*test_*', '*/test*/*', '*.md', 'README.md'])

['*.md', 'README.md']

The include/exclude logic follows the **rsync/grep model**: a file must match at least one `include` pattern (if specified), AND must not match any `exclude` pattern. Exclude always wins—there's no ambiguity. This is simpler and more predictable than gitignore-style ordering rules. Additionally, LLMs are already familiar with this common pattern from tools like `rg` and `rsync`, making it natural to use when this function is provided as an AI tool.

In [ ]:
#| export
def _include(path, include, exclude):
    "Returns True if path matches include patterns (if any) and doesn't match any exclude pattern."
    if include and not any(fnmatch.fnmatch(path, p) for p in listify(include)): return False
    if exclude and any(fnmatch.fnmatch(path, p) for p in listify(exclude)): return False
    return True

With rsync/grep style, exclude always wins. To get "all .md except README.md", you'd include README.md explicitly in your results separately.

In [ ]:
assert not _include('README.md', ['README.md'], ['*.md'])  # exclude wins
assert not _include('CONTRIBUTING.md', ['README.md'], ['*.md'])

Include all .py files except for tests

In [ ]:
assert not _include('examples/test_fastcore2.py', ['*.py'], ['*test_*', '*/test*/*'])
assert not _include('examples/tests/some_test.py', ['*.py'], ['*test_*', '*/tests/*'])
assert not _include('examples/test/some_test.py', ['*.py'], ['*test_*', '*/test/*'])

In [ ]:
assert _include('cool/module.py', ['*.py'], ['setup.py'])
assert not _include('cool/_modidx', ['*.py'], ['*/_modidx'])
assert not _include('setup.py', ['*.py'], ['setup.py'])

In [ ]:
test_repo_files = ['README.md', 'CONTRIBUTING.md', 'dir/MARKDOWN.md', 'tests/file.py',  # chkstyle: ignore-node
    'module/file.py', 'module/app/file.py', 'nbs/00.ipynb', 'file2.py',
    '.gitignore', 'module/.dotfile', '_hidden.py', 'module/_hidden.py']

Here is an example where we filter to include the README, all python files except for the ones under tests directory, include all notebooks, and exclude all files starting with an underscore.

In [ ]:
inc,exc = ['README.md', '*.py', '*.ipynb'], ['tests/*.py', '_*', '*/_*']
[fn for fn in test_repo_files if _include(fn,inc,exc)]

['README.md',
 'module/file.py',
 'module/app/file.py',
 'nbs/00.ipynb',
 'file2.py']

Let's exclude files starting with `test_` and `setup.py` too.

In [ ]:
exc += ['*test_*.py', '*/*test*.py', 'setup.py']
exc

['tests/*.py', '_*', '*/_*', '*test_*.py', '*/*test*.py', 'setup.py']

A function to get repo files with optional filtering

In [ ]:
#| export
@patch
async def _get_repo_files(self:GhApi, owner, repo, branch="main"):
    return await self.git.get_tree(owner=owner, repo=repo, tree_sha=branch, recursive=True)

@patch
async def get_repo_files(self:GhApi, owner, repo, branch="main", inc=None, exc=None):
    "Get all file items of a repo, optionally filtered."
    tree = await self._get_repo_files(owner, repo, branch)
    return L(tree['tree']).filter(lambda o: o['type'] == 'blob' and _include(o.path, inc, exc))

The list of files that are kept based on the filtering logic:

In [ ]:
repo_files = await api.get_repo_files(owner, repo, branch, inc=inc, exc=exc)
test_eq(len(repo_files), 44)

In [ ]:
repo_files.attrgot("path")

['README.md', 'fastcore/all.py', 'fastcore/ansi.py', 'fastcore/basics.py', 'fastcore/dispatch.py', 'fastcore/docments.py', 'fastcore/docscrape.py', 'fastcore/foundation.py', 'fastcore/imghdr.py', 'fastcore/imports.py', 'fastcore/meta.py', 'fastcore/nb_imports.py', 'fastcore/nbio.py', 'fastcore/net.py', 'fastcore/parallel.py', 'fastcore/py2pyi.py', 'fastcore/script.py', 'fastcore/shutil.py', 'fastcore/style.py', 'fastcore/tools.py', 'fastcore/transform.py', 'fastcore/utils.py', 'fastcore/xdg.py', 'fastcore/xml.py', 'fastcore/xtras.py', 'nbs/000_tour.ipynb', 'nbs/00_test.ipynb', 'nbs/01_basics.ipynb', 'nbs/02_foundation.ipynb', 'nbs/03_xtras.ipynb', 'nbs/03a_parallel.ipynb', 'nbs/03b_net.ipynb', 'nbs/04_docments.ipynb', 'nbs/05_meta.ipynb', 'nbs/06_script.ipynb', 'nbs/07_xdg.ipynb', 'nbs/08_style.ipynb', 'nbs/09_xml.ipynb', 'nbs/10_py2pyi.ipynb', 'nbs/11_external.ipynb', 'nbs/12_tools.ipynb', 'nbs/13_nbio.ipynb', 'nbs/index.ipynb', 'tests/minimal.ipynb']

In [ ]:
#| export
@patch
async def get_file_content(self:GhApi, path, owner, repo, branch="main"):
    o = await self.repos.get_content(owner=owner, repo=repo, path=path, ref=branch)
    o['content_decoded'] = base64.b64decode(o.content).decode('utf-8')
    return o

In [ ]:
#| export
@patch
@delegates(GhApi.get_repo_files)
async def get_repo_contents(self:GhApi, owner, repo, branch='main',
    n_workers=16, # Max concurrent downloads
    **kwargs
):
    repo_files = await self.get_repo_files(owner, repo, branch, **kwargs)
    paths = repo_files.attrgot("path")
    return L(await parallel_async(self.get_file_content, paths, owner=owner, repo=repo, branch=branch, n_workers=n_workers))

In [ ]:
contents = await api.get_repo_contents(owner, repo, branch, inc=inc, exc=exc)
md = "\n\n".join(f"**[{o.path}]({o.html_url})**\n```{o.path.split('.')[-1]}\n{chr(10).join(o.content_decoded.split(chr(10))[:5])}\n```" for o in contents[:3])
display(Markdown(md))

<div class="prose" markdown="1">

**[README.md](https://github.com/AnswerDotAI/fastcore/blob/6c22f5d5a79cb8c9edbd51847d2e7b56c92727a3/README.md)**
```md
# Welcome to fastcore


<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

```

**[fastcore/all.py](https://github.com/AnswerDotAI/fastcore/blob/6c22f5d5a79cb8c9edbd51847d2e7b56c92727a3/fastcore/all.py)**
```py
from .imports import *
from .foundation import *
from .utils import *
from .parallel import *
from .net import *
```

**[fastcore/ansi.py](https://github.com/AnswerDotAI/fastcore/blob/6c22f5d5a79cb8c9edbd51847d2e7b56c92727a3/fastcore/ansi.py)**
```py
"Filters for processing ANSI colors."

# Copyright (c) IPython Development Team.
# Modifications by Jeremy Howard.

```

</div>

### GitHub Pages

In [ ]:
#| export
@patch
async def enable_pages(self:GhApi, branch=None, path="/"):
    "Enable or update pages for a repo to point to a `branch` and `path`."
    if path not in ('/docs','/'): raise Exception("path not in ('/docs','/')")
    r = await self.repos.get()
    branch = branch or r.default_branch
    source = {"branch": branch, "path": path}
    if r.has_pages: return # await self.repos.update_information_about_pages_site(source=source)
    if len(await self.list_branches(branch))==0: await self.create_branch_empty(branch)
    return await self.repos.create_pages_site(source=source)

`branch` is set to the default branch if `None`. `path` must be `/docs` or `/`.

In [ ]:
api = GhApi(owner='AnswerDotAI', repo='ghapi-test', token=token)

In [ ]:
res = await api.enable_pages(branch='new-branch', path='/')

test_eq(res.source.branch, 'new-branch')
test_eq(res.source.path, '/')

await api.repos.delete_pages_site()
await api.delete_branch('new-branch')

### Issues and pull requests

Fetching everything needed to review an issue or PR -- title, body, comments, and (for PRs) the diff, inline review comments, and review summaries -- normally takes several separate calls, and the REST API doesn't distinguish PRs from issues for some of them (a PR *is* an issue, so its general comments come from `issues.list_comments`, not `pulls.*`). `read_issue` bundles all of this into one call.

In [ ]:
#| export
class _IssueInfo(AttrDict):
    def _repr_markdown_(self):
        res = [f"**{self.title}** ({'PR' if self.is_pr else 'issue'})", self.body]
        if self.get('diff'): res.append(f"*diff: {self.diff.count('diff --git')} files, {len(self.diff.splitlines())} lines (see `.diff`)*")
        parts = [f"{len(self.comments)} comments"]
        if self.is_pr: parts += [f"{len(self.review_comments)} review comments"] + [f"{v} {k.lower()}" for k,v in Counter(r.state for r in self.reviews).items()]
        res.append(f"*{'; '.join(parts)}*")
        return '\n\n'.join(res)
    __repr__ = _repr_markdown_

@patch
async def read_issue(self:GhApi, issue_number:int):
    "Fetch an issue or PR: title, body, comments, and (for PRs) diff, review comments, and reviews"
    try: pr = await self.pulls.get(issue_number)
    except APIError: pr = None
    iss = await self.issues.get(issue_number)
    res = dict(title=iss.title, body=iss.body or '', is_pr=pr is not None,  # chkstyle: ignore-node
               comments=await self.issues.list_comments(issue_number))
    if pr is not None:
        res['diff'] = await self.pulls.get(issue_number, _headers={'Accept': 'application/vnd.github.v3.diff'})
        res['review_comments'] = await self.pulls.list_review_comments(issue_number)
        res['reviews'] = await self.pulls.list_reviews(issue_number)
    else:
        evts = await self.issues.list_events(issue_number)
        sha = first(e.commit_id for e in evts if e.commit_id)
        if sha: res['diff'] = await self.repos.get_commit(sha, _headers={'Accept': 'application/vnd.github.v3.diff'})
    return _IssueInfo(dict2obj(res))

In [ ]:
api = GhApi(owner='fastai', repo='ghapi', token=token)

pr = await api.read_issue(205)
assert pr.is_pr
assert pr.diff.startswith('diff --git')
test_eq(len(pr.review_comments), 1)
test_eq(len(pr.reviews), 1)

iss = await api.read_issue(206)
assert not iss.is_pr
assert 'diff' in iss and iss.diff.startswith('diff --git')
assert '(issue)' in repr(iss)

r = repr(pr)
assert r.startswith(f'**{pr.title}** (PR)')
assert 'diff --git' not in r
assert '1 review comments' in r
pr

<div class="prose" markdown="1">

**Use Content-Type to determine response parsing** (PR)

Replaces the hardcoded _decode_response endpoint list with Content-Type based response handling.

JSON endpoints return AttrDict, text endpoints return str, binary endpoints return bytes — all determined by the response Content-Type header, not a maintained list of paths.

Supersedes #204.

*diff: 4 files, 190 lines (see `.diff`)*

*0 comments; 1 review comments; 1 commented*

</div>

In [ ]:
#| export
def _filter_diff(diff, folder='', skip_files=('_modidx.py',)):
    "Filter unified diff to only include files under `folder`, skipping `skip_files`"
    sections = re.split(r'(?=^diff --git )', diff, flags=re.MULTILINE)
    return ''.join(s for s in sections  # chkstyle: ignore-node
                   if s.startswith(f'diff --git a/{folder}')
                   and not any(f'/{f} ' in s.split('\n')[0] for f in skip_files))

def _reduce_ctx(diff):
    "Keep only diff headers and changed lines"
    lines = diff.splitlines(True)
    return ''.join(l for l in lines if l[0:1] in ('+','-','d','i','@') or not l.strip())

def _fmt_replies(info):
    "Format an issue/PR's comments, review comments, and reviews (from `GhApi.read_issue`)"
    res = ''
    if info.comments:
        res += '\n\n## Replies\n' + '\n\n---\n'.join(
            f"**@{c.user.login}** ({c.created_at[:10]}):\n{c.body}" for c in info.comments)
    if info.get('review_comments'):
        res += '\n\n## Review comments\n' + '\n\n---\n'.join(
            f"**@{c.user.login}** on `{c.path}:{c.line}` ({c.created_at[:10]}):\n{c.body}" for c in info.review_comments)
    reviews = [r for r in info.get('reviews', []) if r.state != 'PENDING']
    if reviews:
        res += '\n\n## Reviews\n' + '\n\n---\n'.join(
            f"**@{r.user.login}** {r.state} ({(r.submitted_at or '')[:10]}):\n{r.body or ''}" for r in reviews)
    return res

In [ ]:
#| export
async def read_pr(
    pr_number:int|str, # Issue/PR number, or GitHub issue/PR URL
    owner:str=None, # Owner (not needed if URL passed)
    repo:str=None, # Repo (not needed if URL passed)
    folder:str='', # For diffs, limit to only files in `folder`
    replies:bool=False # Include comments, review comments, and reviews?
):
    "Fetch a GitHub PR or issue as one markdown string: title, body, diff (if any), and optionally replies"
    if '/' in str(pr_number): *_,owner,repo,typ,pr_number = str(pr_number).rstrip('/').split('/')
    if not owner or not repo: raise ValueError('Pass `owner` and `repo`, or a full GitHub URL')
    if folder: folder = f"{folder}/"
    info = await GhApi(owner=owner, repo=repo).read_issue(int(pr_number))
    res = f"# {info.title}\n\n{info.body}"
    if info.get('diff'):
        diff = _reduce_ctx(_filter_diff(info.diff, folder=folder))
        res += f"\n\n## Diff\n```diff\n{diff}\n```"
    if replies: res += _fmt_replies(info)
    return res

While `read_issue` returns structured data, `read_pr` formats the whole thing as a single LLM-ready markdown string: title, body, the diff reduced to just headers and changed lines, and (with `replies=True`) comments, inline review comments, and review verdicts. You can pass a number plus `owner`/`repo`, or just paste a full GitHub URL.

In [ ]:
res = await read_pr('https://github.com/fastai/ghapi/pull/205', replies=True)
for s in ('# Use Content-Type', '## Diff', '## Review comments', '## Reviews'): assert s in res

res = await read_pr(206, 'fastai', 'ghapi')
assert '## Diff' in res

with expect_fail(ValueError): await read_pr(205)

`pr_file_diff` is the better choice when you want the complete, untruncated patch for one specific file with addition/deletion counts. `read_pr(folder=...)` is better for getting a reduced overview of all changes in a subdirectory along with the PR context.

In [ ]:
#| export
async def pr_file_diff(
    pr_number:int|str, # Issue/PR number, or GitHub issue/PR URL
    filename:str, # File to get the diff for
    owner:str=None, # Owner (not needed if URL passed)
    repo:str=None, # Repo (not needed if URL passed)
) -> str:
    "Get the untruncated patch/diff for a single file in a PR"
    if '/' in str(pr_number): *_,owner,repo,_,pr_number = str(pr_number).rstrip('/').split('/')
    if not owner or not repo: raise ValueError('Pass `owner` and `repo`, or a full GitHub URL')
    files = await GhApi(owner=owner, repo=repo).pulls.list_files(int(pr_number))
    for f in files:
        if f.filename == filename: return f"## {f.filename} (+{f.additions} -{f.deletions})\n\n```diff\n{html.unescape(f.patch)}\n```"
    return f"File `{filename}` not found in PR #{pr_number}"

In [ ]:
res = await pr_file_diff('https://github.com/fastai/ghapi/pull/205', 'ghapi/core.py')
for s in ('## ghapi/core.py (+9 -18)', '```diff'): assert s in res 

The REST API bypasses issue templates entirely (they're a web-UI feature), so a programmatically-created issue can easily ignore a form the repo requires, and maintainers will bounce it. `issue_template` fetches a repo's templates and parses yml issue forms into their section labels, falling back to the owner-level `.github` community-health repo when the repo has none.

In [ ]:
#| export
def _parse_tmpl(name, txt):
    "Parse issue template `txt`: yml forms into section labels for `issue_body`; md templates kept as `raw`"
    if not name.endswith(('.yml','.yaml')): return dict2obj(dict(name=name, raw=txt))
    import yaml
    d = yaml.safe_load(txt)
    secs = [dict(label=b['attributes']['label'], type=b['type'],  # chkstyle: ignore-node
                 required=bool(nested_idx(b, 'validations', 'required')),
                 options=[o['label'] if isinstance(o, dict) else o for o in b['attributes'].get('options', [])])
            for b in d.get('body', []) if b['type']!='markdown']
    return dict2obj(dict(name=name, title=d.get('name'), description=d.get('description'), sections=secs))

@patch
async def issue_template(self:GhApi):
    "Parsed issue templates for the repo (or the owner's `.github` repo as fallback); pass one to `issue_body`"
    for repo,path in ((None,'.github/ISSUE_TEMPLATE'), ('.github','ISSUE_TEMPLATE'), ('.github','.github/ISSUE_TEMPLATE')):
        kw = dict(repo=repo) if repo else {}
        try: fs = await self.repos.get_content(path=path, **kw)
        except APIError as e:
            if e.status_code != 404: raise
            continue
        return L([_parse_tmpl(f.name, base64.b64decode((await self.repos.get_content(path=f.path, **kw)).content).decode())  # chkstyle: ignore-node
                  for f in fs if f.name.endswith(('.md','.yml','.yaml')) and f.name != 'config.yml'])
    return L()

For example, quarto-cli uses yml issue forms; each parsed template lists the `###` section labels a compliant issue body needs:

In [ ]:
qapi = GhApi(owner='quarto-dev', repo='quarto-cli', token=token)
tmpls = await qapi.issue_template()
bug = first(t for t in tmpls if 'bug' in t.name)
assert bug.sections and all(s.label for s in bug.sections)
test_eq(await GhApi(owner='fastai', repo='ghapi', token=token).issue_template(), [])
[s.label for s in bug.sections]

['I have:',
 'Bug description',
 'Steps to reproduce',
 'Actual behavior',
 'Expected behavior',
 'Your environment',
 'Quarto check output']

In [ ]:
#| export
def issue_body(tmpl, sections):
    "Build an issue body following form `tmpl` from `issue_template`: `### <label>` headings in template order. `sections` maps label to content (for checkbox sections: list of checked options, or `True` for all)"
    if 'raw' in tmpl: raise ValueError(f'{tmpl.name} is a markdown template: adapt `tmpl.raw` directly')
    if (missing := [s.label for s in tmpl.sections if s.required and s.label not in sections]): raise ValueError(f'missing required sections: {missing}')
    if (unknown := set(sections) - {s.label for s in tmpl.sections}): raise ValueError(f'not in template: {unknown}')
    res = []
    for s in tmpl.sections:
        if s.label not in sections: continue
        v = sections[s.label]
        if s.type == 'checkboxes': v = '\n'.join(f"- [{'x' if v is True or o in v else ' '}] {o}" for o in s.options)
        res.append(f'### {s.label}\n\n{v}')
    return '\n\n'.join(res)

`issue_body` then turns a `{label: content}` dict into the body GitHub's own web form would produce, checking required sections and unknown labels so a non-compliant issue fails before it reaches the tracker:

In [ ]:
tmpl = _parse_tmpl('bug_report.yml', """
name: Bug report
description: Report an error
body:
  - type: markdown
    attributes:
      value: Welcome!
  - type: checkboxes
    attributes:
      label: "I have:"
      options:
        - label: searched the issue tracker
        - label: read the docs
  - type: textarea
    attributes:
      label: Bug description
    validations:
      required: true
""")
test_eq([s.label for s in tmpl.sections], ['I have:', 'Bug description'])
body = issue_body(tmpl, {'I have:': True, 'Bug description': 'It breaks.'})
test_eq(body, '### I have:\n\n- [x] searched the issue tracker\n- [x] read the docs\n\n### Bug description\n\nIt breaks.')
body = issue_body(tmpl, {'I have:': ['read the docs'], 'Bug description': 'It breaks.'})
assert '- [ ] searched the issue tracker\n- [x] read the docs' in body
test_fail(lambda: issue_body(tmpl, {'I have:': True}), contains='required')
test_fail(lambda: issue_body(tmpl, {'Bug description': 'x', 'Wrong': 'y'}), contains='template')

### Check / CI status

GitHub Actions results are reported through the Checks API (`checks.list_for_ref`); other CI systems generally use the older Commit Status API (`repos.get_combined_status_for_ref`). Most repos only populate one or the other, so `check_status` merges both into a single result.

In [ ]:
#| export
class _CheckStatus(AttrDict):
    def _repr_markdown_(self):
        runs = self.check_runs
        if not runs: return 'no check runs'
        if any(r.status!='completed' for r in runs): verdict = 'pending'
        elif all(r.conclusion in ('success','neutral','skipped') for r in runs): verdict = 'success'
        else: verdict = 'failure'
        return '\n'.join([f'**{verdict}**', ''] + [f"- {r.name}: {r.conclusion or r.status}" for r in runs])
    __repr__ = _repr_markdown_

@patch
async def check_status(self:GhApi, ref:str):
    "Combined commit status and check-run results for `ref` (a SHA, branch, or tag)"
    combined = await self.repos.get_combined_status_for_ref(ref)
    checks = await self.checks.list_for_ref(ref)
    return _CheckStatus(dict2obj(dict(state=combined.state, statuses=combined.statuses, check_runs=checks.check_runs)))

@patch
async def pr_status(self:GhApi, pull_number:int):
    "Combined status and check-run results for a PR's head commit"
    return await self.check_status((await self.pulls.get(pull_number)).head.sha)

In [ ]:
sha = (await api.actions.list_workflow_runs_for_repo(per_page=1)).workflow_runs[0].head_sha
st = await api.check_status(sha)
assert 'state' in st and 'statuses' in st and 'check_runs' in st
assert len(st.check_runs) > 0

pr_st = await api.pr_status(205)
assert 'state' in pr_st and 'statuses' in pr_st and 'check_runs' in pr_st

test_eq(repr(pr_st), 'no check runs')

r = repr(st)
assert r.startswith('**')
assert all(l.startswith('- ') for l in r.splitlines()[2:])
assert 'html_url' not in r
st

<div class="prose" markdown="1">

**failure**

- build: failure

</div>

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()